# MIDI Preprocessing — Exploratory Notebook

This notebook walks through the MIDI preprocessing pipeline step by step.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import pretty_midi

from src.config import cfg
from src.preprocessing import (
    find_midi_files, midi_to_pianoroll, midi_to_tokens,
    build_pianoroll_dataset, build_token_dataset,
)

## 1. Discover MIDI files

In [ ]:
files = find_midi_files(cfg.RAW_MIDI_DIR)
print(f'Found {len(files)} MIDI files')
print('First 3:', files[:3])

## 2. Inspect one MIDI file

In [ ]:
pm = pretty_midi.PrettyMIDI(files[0])
print(f'Duration: {pm.get_end_time():.2f}s')
print(f'Instruments: {len(pm.instruments)}')
for i, inst in enumerate(pm.instruments):
    print(f'  [{i}] program={inst.program}, n_notes={len(inst.notes)}, drum={inst.is_drum}')

## 3. Piano-roll visualization

In [ ]:
pr = midi_to_pianoroll(files[0])
print(f'Piano-roll shape: {pr.shape}')

plt.figure(figsize=(14, 5))
plt.imshow(pr[:, :500], aspect='auto', origin='lower', cmap='hot')
plt.xlabel('Time step (16 fps)')
plt.ylabel('MIDI pitch')
plt.title(f'Piano-roll: {os.path.basename(files[0])}')
plt.colorbar()
plt.show()

## 4. Tokenization

In [ ]:
toks = midi_to_tokens(files[0])
print(f'Token sequence length: {len(toks)}')
print(f'First 30 tokens: {toks[:30]}')
print(f'Vocab usage: {len(np.unique(toks))} / {cfg.VOCAB_SIZE}')

## 5. Build full datasets

In [ ]:
X_pr,  y_pr  = build_pianoroll_dataset(files[:cfg.MAX_FILES])
X_tok, y_tok = build_token_dataset(files[:cfg.MAX_FILES])

print(f'Piano-roll dataset: X={X_pr.shape}, y={y_pr.shape}')
print(f'Token dataset:      X={X_tok.shape}, y={y_tok.shape}')

# Save processed arrays
os.makedirs(cfg.PROCESSED_DIR, exist_ok=True)
np.savez_compressed(os.path.join(cfg.PROCESSED_DIR, 'pianoroll.npz'), X=X_pr,  y=y_pr)
np.savez_compressed(os.path.join(cfg.PROCESSED_DIR, 'tokens.npz'),    X=X_tok, y=y_tok)
print('Saved processed arrays.')